In [1]:
import os
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import pandas as pd

In [2]:
model_name = "sadickam/sdgBERT"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

In [13]:
def process_discourse(file_path, subfolder):
    # Read the file
    with open(file_path, 'r', encoding='utf-8') as file:
        text = file.read()
    
    # Tokenize and encode the text
    inputs = tokenizer(text, truncation=True, padding=True, return_tensors="pt")
    
    # Get model predictions
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Get probabilities
    probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
    
    # Create a dictionary with results
    results = {
        'Unique_ID': os.path.basename(file_path),
        'Subfolder': subfolder
    }
    
    # Add probabilities for all available SDGs
    for i in range(probs.shape[1]):
        results[f'SDG_{i+1}'] = probs[0][i].item()
    
    return results




In [14]:
# Specify the folder containing discourse files
main_folder = "/Users/mass/Documents/Masters/Courses/Connected Politics/Dataset/dataverse_files/UN General Debate Corpus/subset years"
results = []

In [15]:
# Process all discourse files
for subfolder in os.listdir(main_folder):
    subfolder_path = os.path.join(main_folder, subfolder)
    if os.path.isdir(subfolder_path):
        for file_name in os.listdir(subfolder_path):
            if file_name.endswith('.txt'):
                file_path = os.path.join(subfolder_path, file_name)
                result = process_discourse(file_path, subfolder)
                results.append(result)


In [16]:
# Create a DataFrame from the results
if results:
    df_results = pd.DataFrame(results)
else:
    df_results = pd.DataFrame(columns=['Unique_ID', 'Subfolder'])  # Create empty DataFrame if no results

In [17]:
# Reorder columns to ensure SDGs are in order
sdg_columns = [col for col in df_results.columns if col.startswith('SDG_')]
column_order = ['Unique_ID', 'Subfolder'] + sorted(sdg_columns, key=lambda x: int(x.split('_')[1]))
df_results = df_results[column_order]



In [18]:
# Display the first few rows of the results
print(df_results.head())


         Unique_ID          Subfolder     SDG_1     SDG_2     SDG_3     SDG_4  \
0  BRB_73_2018.txt  Session 73 - 2018  0.003076  0.001269  0.000864  0.003059   
1  IND_73_2018.txt  Session 73 - 2018  0.639088  0.001097  0.001926  0.002821   
2  ARG_73_2018.txt  Session 73 - 2018  0.002001  0.000292  0.000365  0.000558   
3  JOR_73_2018.txt  Session 73 - 2018  0.000104  0.000052  0.000076  0.000103   
4  SWE_73_2018.txt  Session 73 - 2018  0.001305  0.000286  0.000445  0.000935   

      SDG_5     SDG_6     SDG_7     SDG_8     SDG_9    SDG_10    SDG_11  \
0  0.011493  0.001455  0.003482  0.001020  0.001699  0.001412  0.005963   
1  0.181816  0.000591  0.001478  0.013735  0.001929  0.133095  0.001418   
2  0.050529  0.000186  0.000205  0.000265  0.000210  0.001959  0.000273   
3  0.000203  0.000056  0.000051  0.000028  0.000040  0.000284  0.000056   
4  0.753481  0.000281  0.000393  0.000651  0.000391  0.001797  0.000351   

     SDG_12    SDG_13    SDG_14    SDG_15    SDG_16  
0  0.001

In [19]:
len(df_results)

3096

In [23]:
df_results["Subfolder"].str[-4:]


0       2018
1       2018
2       2018
3       2018
4       2018
        ... 
3091    2014
3092    2014
3093    2014
3094    2014
3095    2014
Name: Subfolder, Length: 3096, dtype: object

In [24]:
df_results["Unique_ID"] = df_results["Unique_ID"].str.replace(".txt", "")
df_results["Subfolder"] = df_results["Subfolder"].str[-4:]


In [26]:
df_results.rename(columns={"Unique_ID": "Id", "Subfolder": "Year"}, inplace=True)

In [27]:
df_results.head()

,Id,Year,SDG_1,SDG_2,SDG_3,SDG_4,SDG_5,SDG_6,SDG_7,SDG_8,SDG_9,SDG_10,SDG_11,SDG_12,SDG_13,SDG_14,SDG_15,SDG_16
0,BRB_73_2018,2018,0.003076,0.001269,0.000864,0.003059,0.011493,0.001455,0.003482,0.001020,0.001699,0.001412,0.005963,0.001301,0.954183,0.001333,0.004044,0.004347
1,IND_73_2018,2018,0.639088,0.001097,0.001926,0.002821,0.181816,0.000591,0.001478,0.013735,0.001929,0.133095,0.001418,0.001058,0.002069,0.001110,0.000637,0.016131
2,ARG_73_2018,2018,0.002001,0.000292,0.000365,0.000558,0.050529,0.000186,0.000205,0.000265,0.000210,0.001959,0.000273,0.000098,0.000284,0.000544,0.000225,0.942004
3,JOR_73_2018,2018,0.000104,0.000052,0.000076,0.000103,0.000203,0.000056,0.000051,0.000028,0.000040,0.000284,0.000056,0.000035,0.000055,0.000123,0.000057,0.998676
4,SWE_73_2018,2018,0.001305,0.000286,0.000445,0.000935,0.753481,0.000281,0.000393,0.000651,0.000391,0.001797,0.000351,0.000128,0.000562,0.000852,0.000556,0.237586


In [28]:
#Import speakers by session

speakers = pd.read_excel("/Users/mass/Documents/Masters/Courses/Connected Politics/Dataset/dataverse_files/UN General Debate Corpus/Speakers_by_session.xlsx")
speakers.head()

,Year,Session,ISO Code,Country,Name of Person Speaking,Post,ID
0,2023,78,BRA,Brazil,Luiz Inacio Lula da Silva,President,BRA_78_2023
1,2023,78,USA,United States of America,Joseph R. Biden,President,USA_78_2023
2,2023,78,COL,Colombia,Gustavo Petro Urrego,President,COL_78_2023
3,2023,78,JOR,Jordan,Abdullah II ibn Al Hussein,King,JOR_78_2023
4,2023,78,POL,Poland,Andrzej Duda,President,POL_78_2023


In [31]:
df_results = df_results.merge(speakers[["ID", "Post"]], left_on = "Id", right_on = "ID", how = "left")

In [34]:
df_results = df_results[['Id', 'Year', 'Post', 'SDG_1', 'SDG_2', 'SDG_3', 'SDG_4', 'SDG_5', 'SDG_6',
                         'SDG_7', 'SDG_8', 'SDG_9', 'SDG_10', 'SDG_11', 'SDG_12', 'SDG_13',
                         'SDG_14', 'SDG_15', 'SDG_16']]

In [36]:
df_results.head()

,Id,Year,Post,SDG_1,SDG_2,SDG_3,SDG_4,SDG_5,SDG_6,SDG_7,SDG_8,SDG_9,SDG_10,SDG_11,SDG_12,SDG_13,SDG_14,SDG_15,SDG_16
0,BRB_73_2018,2018,"Prime Minister, Minister for National Security...",0.003076,0.001269,0.000864,0.003059,0.011493,0.001455,0.003482,0.001020,0.001699,0.001412,0.005963,0.001301,0.954183,0.001333,0.004044,0.004347
1,IND_73_2018,2018,Minister for External Affairs,0.639088,0.001097,0.001926,0.002821,0.181816,0.000591,0.001478,0.013735,0.001929,0.133095,0.001418,0.001058,0.002069,0.001110,0.000637,0.016131
2,ARG_73_2018,2018,President,0.002001,0.000292,0.000365,0.000558,0.050529,0.000186,0.000205,0.000265,0.000210,0.001959,0.000273,0.000098,0.000284,0.000544,0.000225,0.942004
3,JOR_73_2018,2018,King,0.000104,0.000052,0.000076,0.000103,0.000203,0.000056,0.000051,0.000028,0.000040,0.000284,0.000056,0.000035,0.000055,0.000123,0.000057,0.998676
4,SWE_73_2018,2018,Chair of the Delegation,0.001305,0.000286,0.000445,0.000935,0.753481,0.000281,0.000393,0.000651,0.000391,0.001797,0.000351,0.000128,0.000562,0.000852,0.000556,0.237586


In [37]:
len(df_results)

3096

In [35]:

# Save the results to a CSV file
df_results.to_csv('/Users/mass/Documents/Masters/Courses/Connected Politics/Final dataset/sdg_analysis_results.csv', index = False)